In [4]:
%run libs.py
from pathlib import Path

In [5]:
%run universal-framework.ipynb

In [6]:
LSTM_EPOCHS = 1
LSTM_LOOKBACK = 8

NHITS_MAX_STEPS = 100

In [7]:
def create_windows(
    y,
    lookback=24
):

    X = []
    target = []

    for i in range(
        lookback,
        len(y)
    ):

        X.append(
            y[i - lookback:i]
        )

        target.append(
            y[i]
        )

    X = np.array(X)
    target = np.array(target)

    X = X.reshape(
        X.shape[0],
        X.shape[1],
        1
    )

    return X, target

In [8]:
def recursive_lstm_forecast(
    model,
    history,
    horizon,
    lookback
):

    history = list(history)

    forecasts = []

    for _ in range(horizon):

        x = np.array(
            history[-lookback:]
        ).reshape(
            1,
            lookback,
            1
        )

        pred = model.predict(
            x,
            verbose=0
        )[0][0]

        forecasts.append(pred)

        history.append(pred)

    return np.array(forecasts)

In [9]:
def lstm_forecast(
    train,
    horizon,
    lookback=LSTM_LOOKBACK
):

    if len(train) < lookback + horizon:
        return np.repeat(
            train[-1],
            horizon
        )

    X, y = create_windows(
        train,
        lookback
    )

    model = Sequential([
        LSTM(32),
        Dense(1)
    ])

    model.compile(
        optimizer="adam",
        loss="mse"
    )

    model.fit(
        X,
        y,
        epochs=LSTM_EPOCHS,
        batch_size=32,
        verbose=0
    )

    return recursive_lstm_forecast(
        model=model,
        history=train,
        horizon=horizon,
        lookback=lookback
    )

In [ ]:
lstm_results = run_benchmark(
    model_name="LSTM",
    forecast_function=lstm_forecast,
    output_file=RESULTS_DIR / "lstm.csv"
)


Running LSTM on std_agri_medium
Completed std_agri_medium in 867.9s

Running LSTM on std_agri_small
Completed std_agri_small in 172.0s

Running LSTM on std_energy_medium


In [ ]:
nhits_results = []

for file in DATA_DIR.glob("*.csv"):

    dataset_name = file.stem

    print(
        f"\nRunning*NHITS on {dataset_name}"
    )

    try:

        raw_df = pd.read_csv(
            file,
            low_memory=False
        )

        df, freq = (
            prepare_neuralforecast_dataset(
                raw_df,
                dataset_name
            )
        )

        print(
            f"Series={df['unique_id'].nunique()}, "
            f"Rows={len(df):,}, "
            f"Freq={freq}"
        )

    except Exception as e:

        print(             
            f"{dataset_name}: preprocessing failed -> {e}"
        )
        continue

    for horizon,*horizon_type in get_horizons_info(        
        dataset_name
    ):

        try:

            start = time.time()

            model = NHITS(
                h=horizon,
                input_size=min(
                    max(
                        horizon * 2,
                        horizon
                    ),
                    128
                ),
                max_steps=N*ITS_MAX_STEPS
            )

            nf = NeuralForecast(
                models=[model],
                freq=freq
            )

            nf.fit(df)

            forecasts = nf.predict()

            runtime = (
                time.time() - start
            )

            nhits_results.append({

                "dataset":
                    dataset_name,

                "model":
                    "NHITS",

                "horizon":
                    horizon,

                "horizon_type":
                    horizon_type,

                "frequency":
                    freq,

                "n_series":
                    df["unique_id"].nunique(),

                "n_rows":
                    len(df),

                "runtime_seconds":
                    runtime

            })

            pd.DataFrame(
                nhits_results
            ).to_csv(
                RESULTS_DIR / "nhits.csv",
                index=False
            )

        except Exception as e:

            print(
                f"{dataset_name} "
                f"({horizon_type}): {e}"
            )

    del raw_df
    del df

    gc.collect()

In [ ]:
nhits_results = pd.DataFrame(
    nhits_results
)

display(
    nhits_results.head()
)

In [ ]:
display(
    lstm_results.groupby(
        "horizon_type"
    )[
        ["MAE", "RMSE", "SMAPE"]
    ].mean()
)

if len(nhits_results):

    display(
        nhits_results.groupby(
            "horizon_type"
        )[
            ["runtime_seconds"]
        ].mean()
    )